[![](imagens/colab-badge.png){width="16%"}](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cap06/cap06.EPs_aluno.ipynb)
[![](imagens/github-badge.png){width="19%"}](https://github.com/fzampirolli/pdi-vc)


## 💻 Parte Prática com Exercícios de Programação

Os exercícios de programação (EP) desta seção complementam os conceitos apresentados ao longo do Capítulo 6 por meio da implementação de algoritmos relacionados à inspeção industrial e à análise de documentos. O objetivo é consolidar os fundamentos estudados, reproduzindo, em escala reduzida, etapas de um *pipeline* típico de Visão Computacional.

Diferentemente dos capítulos anteriores, cujos exercícios enfatizavam operações mais diretamente relacionadas aos dados de imagem, os EPs deste capítulo concentram-se nas **grandezas intermediárias** produzidas durante o processamento, como áreas, perímetros, circularidade, ângulos de retas, graus de preenchimento de bolhas, mapas de variância e mapas de diferença. Essa abordagem permite compreender e validar cada etapa do *pipeline* de forma independente, sem depender de bibliotecas especializadas para aquisição de imagens, detecção de marcadores ou decodificação de códigos — com exceção do exercício de encerramento do capítulo (EP06_08), que propositalmente introduz o uso do OpenCV para a segmentação e a decodificação real de um *QRCode*, fechando o ciclo entre os conceitos teóricos e as ferramentas empregadas na prática.

Os exercícios seguem a mesma sequência conceitual do capítulo, em ordem crescente de complexidade. Inicialmente, são abordadas métricas de avaliação de segmentação, utilizadas para quantificar a qualidade de máscaras binárias. Em seguida, estudam-se critérios geométricos para seleção de marcadores, classificação de marcações em formulários e estimação da inclinação de documentos por meio da Transformada de Hough. Na parte final, os exercícios exploram a normalização de iluminação, a detecção de defeitos por análise de textura e a integração entre registro geométrico e subtração de imagens em um *pipeline* simplificado de inspeção industrial.

Cada exercício representa uma etapa isolada de um sistema real de Visão Computacional, permitindo validar individualmente conceitos que, em aplicações industriais, são combinados em um único *pipeline* de inspeção.

### 🗺️ Legenda de Dificuldade {.unnumbered}

| Nível | Significado | EPs |
|:---:|---|---|
| 🟢 | Muito fácil / fácil — implementação de um único conceito ou algoritmo simples | EP06_01, EP06_02 |
| 🟡 | Fácil–médio — tratamento de múltiplos casos ou utilização de critérios estatísticos simples | EP06_03, EP06_04 |
| 🟠 | Médio — processamento matricial ponto a ponto | EP06_05 |
| 🔴 | Difícil — processamento matricial com operações em vizinhança (janela deslizante) | EP06_06 |
| 🟣 | Muito difícil — integração de múltiplas etapas de um *pipeline* de Visão Computacional | EP06_07 |
| ⚫ | Especial — uso de biblioteca especializada (`cv2`) para segmentação geométrica e decodificação real de código de barras/QRCode | EP06_08 |

::: {.callout-important}
### Diretrizes para a Resolução dos Exercícios de Programação {.unnumbered}

Salvo indicação em contrário, todos os exercícios utilizam a convenção de coordenadas matriciais `[linha][coluna]`, com origem em $(0,0)$ no canto superior esquerdo da imagem.

Quando houver necessidade de arredondamento numérico, deve-se utilizar o arredondamento padrão para o inteiro mais próximo (*round half away from zero*, com `np.floor(img + 0.5)`). Comparações com limiares (por exemplo, circularidade, variância, diferença de intensidade ou grau de preenchimento) devem ser consideradas **estritas** (`>`), exceto quando o enunciado especificar explicitamente outro critério.

Cada exercício foi elaborado para enfatizar um conceito específico apresentado no capítulo. Recomenda-se implementar inicialmente a solução de forma direta e, somente após sua validação, buscar alternativas mais eficientes ou mais gerais.
:::

### 🎯 Objetivo deste Caderno {.unnumbered}

Este caderno foi elaborado para apoiar o desenvolvimento, a validação e os testes das soluções dos **Exercícios de Programação (EPs)** em um ambiente interativo, como o Google Colab ou o Jupyter Notebook. Após verificar o funcionamento da implementação com os casos de teste apresentados, o código pode ser submetido ao Moodle para a avaliação oficial.

#### *Download* {.unnumbered}

Execute a célula a seguir para obter os arquivos `morph.py` e `testsuite.py`, utilizados pelos exercícios deste capítulo.

In [1]:
import os, sys, importlib, inspect, urllib.request

# URLs do repositório
BASE_URL = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph"
for f in ["morph.py", "testsuite.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{BASE_URL}/{f}", f)

import morph, testsuite
importlib.reload(morph); importlib.reload(testsuite)
from morph import mm
from testsuite import TestSuite

print(f"✅ Ambiente pronto. Morph: {morph.__version__} | TestSuite: {testsuite.__version__}")


✅ Ambiente pronto. Morph: 1.1.2 | TestSuite: 1.1.2


#### Executando os Testes {.unnumbered}

Após implementar a solução, execute `TestSuite("EP06_01.extensão").run()` em uma nova célula, substituindo `extensão` pela linguagem utilizada (`.py`, `.java`, `.c`, `.cpp`, `.js` ou `.r`). O sistema obtém automaticamente os casos de teste do repositório do curso, executa o programa e apresenta o resultado da avaliação.

Em Python, também é possível testar a solução diretamente a partir de uma *string*, sem a necessidade de salvar o código em um arquivo. Para isso, armazene o programa em uma variável e utilize o método `run_code`:

```python
codigo = """
# ... seu código aqui ...
"""

TestSuite("EP06_01").run_code(codigo)
```

### EP06_01 🟢 Avaliação de Segmentação por IoU (*Intersection over Union*)

Ao longo deste capítulo, diversas etapas do *pipeline* produzem **máscaras binárias**, como na segmentação de documentos, na localização de *QRCodes* e na detecção de defeitos. Para avaliar objetivamente a qualidade dessas segmentações, é necessário compará-las com uma máscara de referência (*ground truth*).

Uma das métricas mais utilizadas para esse fim é a **IoU** (*Intersection over Union*, ou Interseção sobre União), definida como a razão entre a área de interseção e a área de união de duas máscaras binárias. Quanto maior o valor da IoU, maior a concordância entre a segmentação produzida pelo algoritmo e a referência.

#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (número de linhas) e $C$ (número de colunas).
2. **Máscara de referência:** Ler os $L \times C$ elementos binários (0 ou 1) da matriz `ref`.
3. **Máscara predita:** Ler os $L \times C$ elementos binários (0 ou 1) da matriz `pred`.
4. **Interseção:** Contar o número de posições $(i,j)$ para as quais `ref[i][j] = 1` e `pred[i][j] = 1`.
5. **União:** Contar o número de posições $(i,j)$ para as quais `ref[i][j] = 1` ou `pred[i][j] = 1`.
6. **Caso degenerado:** Se a união for igual a $0$, definir $\mathrm{IoU}=1{,}0$, pois ambas as máscaras são vazias.
7. **Cálculo:** Caso a união seja maior que zero, calcular

$$
\mathrm{IoU}=
\frac{|\mathrm{Intersecao}|}
{|\mathrm{Uniao}|}.
$$

8. **Classificação:** Determinar a classificação qualitativa utilizando o valor de IoU **antes** do arredondamento.
9. **Arredondamento:** Exibir a IoU com quatro casas decimais.
10. **Saída:** Imprimir, nessa ordem, a interseção, a união, a IoU e a classificação.

#### 📌 Restrições Computacionais

- Se a união for igual a $0$, não deve ser realizada a divisão; a IoU deve ser definida como $1{,}0$.
- As faixas de classificação utilizam comparações não estritas ($\geq$).
- A classificação deve ser realizada utilizando o valor da IoU em precisão completa, antes do arredondamento para exibição.

#### 🧠 Fundamentação Teórica

A IoU é definida por

$$
\mathrm{IoU}=
\frac{|R\cap P|}
{|R\cup P|},
$$

em que:

- $R$ representa o conjunto de pixels pertencentes à máscara de referência;
- $P$ representa o conjunto de pixels pertencentes à máscara predita;
- $|R\cap P|$ corresponde ao número de pixels pertencentes simultaneamente às duas máscaras;
- $|R\cup P|$ corresponde ao número de pixels pertencentes a pelo menos uma das máscaras.

| Faixa de IoU | Classificação | Interpretação |
|---|---|---|
| $\mathrm{IoU}\geq0{,}90$ | `EXCELENTE` | Concordância muito elevada entre as máscaras. |
| $0{,}70\leq\mathrm{IoU}<0{,}90$ | `BOM` | Pequenas diferenças entre as máscaras. |
| $0{,}50\leq\mathrm{IoU}<0{,}70$ | `ACEITAVEL` | Concordância parcial entre as máscaras. |
| $\mathrm{IoU}<0{,}50$ | `RUIM` | Baixa concordância entre as máscaras. |

A IoU depende apenas da sobreposição entre as máscaras e, portanto, é independente do tamanho da imagem.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

- Linha 1: inteiro $L$.
- Linha 2: inteiro $C$.
- Próximas $L$ linhas: elementos binários (0 ou 1) da matriz `ref`.
- Próximas $L$ linhas: elementos binários (0 ou 1) da matriz `pred`.

**Saída:**

- Linha 1: `Intersecao: X`
- Linha 2: `Uniao: Y`
- Linha 3: `IoU: Z`
- Linha 4: `Classificacao: NOME`

O valor de `IoU` deve ser impresso com quatro casas decimais.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 2<br>2<br>1 1<br>0 0<br>1 0<br>0 0 | Intersecao: 1<br>Uniao: 2<br>IoU: 0.5000<br>Classificacao: ACEITAVEL | A metade da região de referência foi corretamente segmentada. |
| 2<br>2<br>0 0<br>0 0<br>0 0<br>0 0 | Intersecao: 0<br>Uniao: 0<br>IoU: 1.0000<br>Classificacao: EXCELENTE | Ambas as máscaras são vazias; por convenção, $\mathrm{IoU}=1{,}0$. |

In [1]:
#| label: fig-06-sim-ep01
#| fig-cap: "Simulador: IoU entre máscara de referência e máscara predita"
#| echo: false
#| output: true

from IPython.display import HTML

HTML("""
<div id="sim06_ep01" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim06_ep01 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim06_ep01 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim06_ep01 button:hover { background: #e8dfcf; }
  #sim06_ep01 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim06_ep01_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim06_ep01_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim06_ep01_grid_ctrls { display: grid; grid-template-columns: repeat(auto-fit, minmax(140px, 1fr)); gap: 12px; }
  .sim06_ep01_px { width: 24px; height: 24px; border: 1px solid #e4dcc8; box-sizing: border-border; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP06_01: IoU (Intersection over Union)</span>
  <span class="sim06_ep01_pill">IoU = |A &cap; B| / |A &cup; B|</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Controles -->
  <div class="sim06_ep01_panel" style="margin-bottom:14px;">
    <div class="sim06_ep01_grid_ctrls">
      
      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Deslocamento H (&Delta;x)</label>
          <span id="sim06_ep01_vdx" style="font-family:monospace; font-weight:700; color:#26241d;">0</span>
        </div>
        <input id="sim06_ep01_dx" type="range" min="-3" max="3" step="1" value="0">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Deslocamento V (&Delta;y)</label>
          <span id="sim06_ep01_vdy" style="font-family:monospace; font-weight:700; color:#26241d;">0</span>
        </div>
        <input id="sim06_ep01_dy" type="range" min="-3" max="3" step="1" value="0">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Lado do Quadrado</label>
          <span id="sim06_ep01_vsz" style="font-family:monospace; font-weight:700; color:#26241d;">6</span>
        </div>
        <input id="sim06_ep01_sz" type="range" min="2" max="8" step="1" value="6">
      </div>

    </div>

    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:10px; text-align:center;">
      Desloque e redimensione a máscara predita para avaliar o alinhamento.
    </div>
  </div>

  <!-- Exibição das Máscaras 10x10 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); gap:12px; margin-bottom:14px;">
    
    <div class="sim06_ep01_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        Referência (A)
      </div>
      <div id="sim06_ep01_ref" style="display:grid; grid-template-columns:repeat(10, 24px); gap:2px; justify-content:center;"></div>
    </div>

    <div class="sim06_ep01_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        Predita (B)
      </div>
      <div id="sim06_ep01_pred" style="display:grid; grid-template-columns:repeat(10, 24px); gap:2px; justify-content:center;"></div>
    </div>

    <div class="sim06_ep01_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        Sobreposição (A &cap; B)
      </div>
      <div id="sim06_ep01_mix" style="display:grid; grid-template-columns:repeat(10, 24px); gap:2px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim06_ep01_dbg" class="sim06_ep01_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep01(root){
    if (!root || root.dataset.sim06Ep01Init) return;
    root.dataset.sim06Ep01Init = "1";

    var dx = root.querySelector("#sim06_ep01_dx");
    var dy = root.querySelector("#sim06_ep01_dy");
    var sz = root.querySelector("#sim06_ep01_sz");

    var vdx = root.querySelector("#sim06_ep01_vdx");
    var vdy = root.querySelector("#sim06_ep01_vdy");
    var vsz = root.querySelector("#sim06_ep01_vsz");

    var gRef  = root.querySelector("#sim06_ep01_ref");
    var gPred = root.querySelector("#sim06_ep01_pred");
    var gMix  = root.querySelector("#sim06_ep01_mix");

    var dbg = root.querySelector("#sim06_ep01_dbg");

    var N = 10;
    var ref = {x: 2, y: 2, w: 6, h: 6};

    function inside(x, y, r){
      return x >= r.x && x < r.x + r.w && y >= r.y && y < r.y + r.h;
    }

    function pixel(color){
      var d = document.createElement("div");
      d.className = "sim06_ep01_px";
      d.style.background = color;
      return d;
    }

    function classe(i){
      if (i >= 0.90) return "Excelente";
      if (i >= 0.75) return "Muito boa";
      if (i >= 0.50) return "Aceitável";
      return "Ruim";
    }

    function render(){
      vdx.textContent = dx.value;
      vdy.textContent = dy.value;
      vsz.textContent = sz.value;

      gRef.innerHTML  = "";
      gPred.innerHTML = "";
      gMix.innerHTML  = "";

      var pred = {
        x: ref.x + parseInt(dx.value, 10),
        y: ref.y + parseInt(dy.value, 10),
        w: parseInt(sz.value, 10),
        h: parseInt(sz.value, 10)
      };

      var inter = 0;
      var uniao = 0;

      for (var y = 0; y < N; y++){
        for (var x = 0; x < N; x++){
          var r = inside(x, y, ref);
          var p = inside(x, y, pred);

          gRef.appendChild(pixel(r ? "#7fdc92" : "#ffffff"));
          gPred.appendChild(pixel(p ? "#7fbfff" : "#ffffff"));

          if (r && p){
            gMix.appendChild(pixel("#9b59b6"));
            inter++;
          }
          else if (r){
            gMix.appendChild(pixel("#7fdc92"));
            uniao++;
          }
          else if (p){
            gMix.appendChild(pixel("#7fbfff"));
            uniao++;
          }
          else{
            gMix.appendChild(pixel("#ffffff"));
          }

          if (r && p) uniao++;
        }
      }

      var iou = inter / uniao;

      dbg.innerHTML =
        "<b>Interseção</b> = " + inter + " pixels &nbsp;&nbsp;&nbsp;" +
        "<b>União</b> = " + uniao + " pixels<br><br>" +
        "IoU = <b>" + inter + " / " + uniao + " = " + iou.toFixed(4) + "</b><br><br>" +
        "<span style='font-weight:700; color:#04342C;'>" + classe(iou) + "</span>";
    }

    dx.addEventListener('input', render);
    dy.addEventListener('input', render);
    sz.addEventListener('input', render);

    render();
  }

  function tryInitSim06Ep01(){
    var root = document.getElementById('sim06_ep01');
    if (root) initSim06Ep01(root); else setTimeout(tryInitSim06Ep01, 200);
  }
  tryInitSim06Ep01();
})();
</script>
""")

In [3]:
%%writefile EP06_01.py
# Código Python

Overwriting EP06_01.py


In [4]:
TestSuite("EP06_01.py").run()

### EP06_02 🟢 Filtro de Marcadores por Circularidade

Após a segmentação de uma imagem, é comum que diversos componentes conexos sejam identificados. Em aplicações como a retificação de documentos, apenas alguns desses componentes correspondem aos marcadores de referência utilizados para o alinhamento da imagem. Um critério frequentemente empregado para selecionar esses marcadores é a **circularidade**, que mede o quão próxima a forma de um componente está de um círculo.

Neste exercício, cada componente é descrito por sua área $A$ e seu perímetro $P$. O objetivo é calcular sua circularidade e decidir, a partir de um limiar fornecido, se o componente deve ser aceito ou rejeitado como candidato a marcador.

#### 📋 Diretrizes de Implementação

1. **Quantidade:** Ler o inteiro $N$ (número de candidatos) e o limiar de circularidade $C_{\text{limiar}}$ (número real).
2. **Dados dos candidatos:** Para cada um dos $N$ candidatos, ler a área $A$ (inteiro) e o perímetro $P$ (número real).
3. **Circularidade:** Calcular $C=\frac{4\pi A}{P^2}$, em que:

- $A$ é a área do componente;
- $P$ é o perímetro do componente;
- $C$ é a circularidade.

4. **Caso degenerado:** Se $P=0$, considerar $C=0$ e classificar diretamente o candidato como `REJEITADO`.
5. **Classificação:** Se $C>C_{\text{limiar}}$, classificar o candidato como `ACEITO`; caso contrário, classificá-lo como `REJEITADO`.
6. **Arredondamento:** Exibir o valor de $C$ com quatro casas decimais.
7. **Saída:** Para cada candidato, imprimir o valor de $C$ seguido da classificação. Ao final, imprimir o número total de candidatos aceitos.

#### 📌 Restrições Computacionais

- Utilizar a constante $\pi$ da biblioteca padrão da linguagem (por exemplo, `math.pi`), sem aproximações.
- A comparação deve ser realizada com o valor de $C$ em precisão completa, antes do arredondamento para exibição.
- O critério de aceitação é estrito ($C>C_{\text{limiar}}$).
- Se $P=0$, a divisão não deve ser realizada.

#### 🧠 Fundamentação Teórica

A circularidade é um descritor geométrico definido por $C=\frac{4\pi A}{P^2}$, em que:

- $A$ é a área do componente;
- $P$ é o perímetro do componente;
- $C$ é a circularidade.

Para um círculo perfeito, $C=1$. À medida que a forma se torna mais alongada ou irregular, o perímetro cresce mais rapidamente que a área, reduzindo o valor de $C$.

| Forma | Circularidade aproximada | Interpretação |
|---|---:|---|
| Círculo | $1{,}0000$ | Forma circular. |
| Quadrado | $0{,}7854$ | Forma aproximadamente compacta. |
| Forma alongada ou irregular | $C\ll1$ | Baixa circularidade. |
| $P=0$ | $0$ (convenção adotada) | Contorno degenerado. |

A circularidade é invariante à translação, à rotação e à escala, sendo amplamente utilizada para distinguir componentes aproximadamente circulares de outros formatos.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

- Linha 1: inteiro $N$.
- Linha 2: número real $C_{\text{limiar}}$.
- Próximas $N$ linhas: área $A$ (inteiro) e perímetro $P$ (real), separados por espaço.

**Saída:**

- Uma linha para cada candidato, no formato `C ACEITO` ou `C REJEITADO`, com $C$ apresentado com quatro casas decimais.
- Última linha: `Total aceitos: X`.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 3<br>0.6<br>78 31.4<br>100 40<br>50 60 | 0.9941 ACEITO<br>0.7854 ACEITO<br>0.1745 REJEITADO<br>Total aceitos: 2 | Candidato aproximadamente circular, forma compacta e forma alongada. |
| 1<br>0.9<br>10 0 | 0.0000 REJEITADO<br>Total aceitos: 0 | Perímetro nulo: contorno degenerado. |

In [2]:
#| label: fig-06-sim-ep02
#| fig-cap: "Simulador: Filtro de Marcadores por Circularidade"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim06_ep02" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim06_ep02 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim06_ep02 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim06_ep02 button:hover { background: #e8dfcf; }
  #sim06_ep02 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim06_ep02_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim06_ep02_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP06_02: Filtro de Marcadores por Circularidade</span>
  <span class="sim06_ep02_pill">C = 4&pi;A / P&sup2;</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim06_ep02_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Limiar de Circularidade (C_limiar): <span id="sim06_ep02_vl" style="font-family:monospace; color:#26241d;">0.60</span>
      </label>
    </div>
    
    <input id="sim06_ep02_sl" type="range" min="0.05" max="0.99" step="0.01" value="0.60">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajuste o limiar e observe quais candidatos (discos, quadrados e formas irregulares) sobrevivem ao filtro.
    </div>
  </div>

  <!-- Cards de Candidatos -->
  <div id="sim06_ep02_cards" style="display:grid; grid-template-columns: repeat(auto-fit, minmax(100px, 1fr)); gap:10px; margin-bottom:14px;"></div>

  <!-- Painel Informativo / Debug -->
  <div id="sim06_ep02_debug" class="sim06_ep02_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep02(root){
    if (!root || root.dataset.sim06Ep02Init) return;
    root.dataset.sim06Ep02Init = "1";

    var candidatos = [
      {nome: "Disco", A: 78, P: 31.4},
      {nome: "Quadrado", A: 100, P: 40},
      {nome: "Retângulo", A: 60, P: 44},
      {nome: "Rasura", A: 50, P: 60},
      {nome: "Ponto", A: 10, P: 0}
    ];

    var slEl  = root.querySelector('#sim06_ep02_sl');
    var vlEl  = root.querySelector('#sim06_ep02_vl');
    var cards = root.querySelector('#sim06_ep02_cards');
    var dbg   = root.querySelector('#sim06_ep02_debug');

    function render(){
      var th = parseFloat(slEl.value);
      vlEl.textContent = th.toFixed(2);
      cards.innerHTML = '';
      var aceitos = 0;

      candidatos.forEach(function(c){
        var C = (c.P === 0) ? 0 : (4 * Math.PI * c.A) / (c.P * c.P);
        var ok = c.P !== 0 && C > th;
        if (ok) aceitos++;

        var div = document.createElement('div');
        div.style.cssText = 'text-align:center; border-radius:10px; padding:10px 6px; font-size:11px; transition:all 0.15s ease;' +
          (ok ? 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;' : 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;');
        
        div.innerHTML = '<div style="font-weight:700; margin-bottom:4px;">' + c.nome + '</div>' +
          '<div style="font-family:monospace; margin-bottom:4px; font-size:10px; opacity:0.8;">A = ' + c.A + '<br>P = ' + c.P + '</div>' +
          '<div style="font-family:monospace; font-weight:700; margin-bottom:4px;">C = ' + C.toFixed(4) + '</div>' +
          '<div style="font-weight:700; font-size:10px; letter-spacing:0.04em;">' + (ok ? 'ACEITO' : 'REJEITADO') + '</div>';
        
        cards.appendChild(div);
      });

      dbg.textContent = 'C_limiar = ' + th.toFixed(2) + '  |  Candidatos aceitos: ' + aceitos + ' / ' + candidatos.length;
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep02(){
    var root = document.getElementById('sim06_ep02');
    if (root) initSim06Ep02(root); else setTimeout(tryInitSim06Ep02, 200);
  }
  tryInitSim06Ep02();
})();
</script>
""")

In [6]:
%%writefile EP06_02.py
# Código Python

Overwriting EP06_02.py


In [7]:
TestSuite("EP06_02.py").run()

### EP06_03 🟡 Classificação de Marcações em Folhas de Resposta (OMR)

Após a retificação da folha e a segmentação dos quadros de respostas, o MCTest estima, para cada bolha, um **grau de preenchimento**, representado por um valor entre $0$ e $100$. A partir desses valores, o sistema deve determinar automaticamente a alternativa marcada, identificando também questões em branco e casos de múltiplas marcações.

Neste exercício, você implementará essa etapa de decisão do *pipeline* de OMR. A classificação depende de um limiar de preenchimento: pequenas variações nesse valor podem alterar o resultado da leitura automática.

#### 📋 Diretrizes de Implementação

1. **Parâmetros:** Ler os inteiros $Q$ (número de questões) e $K$ (número de alternativas por questão, com $2 \le K \le 26$) e o limiar de preenchimento $\mathrm{Th}$ (número real entre $0$ e $100$).
2. **Graus de preenchimento:** Para cada uma das $Q$ questões, ler os $K$ valores reais correspondentes às alternativas `A`, `B`, `C`, ..., na ordem de entrada.
3. **Contagem de marcações:** Para cada questão, contar quantas alternativas possuem grau de preenchimento **estritamente maior** que $\mathrm{Th}$.
4. **Classificação:**
   - Se nenhuma alternativa exceder $\mathrm{Th}$, classificar a questão como `BRANCO`.
   - Se exatamente uma alternativa exceder $\mathrm{Th}$, imprimir a letra correspondente (`A`, `B`, `C`, ...).
   - Se duas ou mais alternativas excederem $\mathrm{Th}$, classificar a questão como `DUPLA_MARCACAO`.
5. **Saída por questão:** Imprimir, na ordem de leitura, a classificação de cada questão.
6. **Totais:** Ao final, imprimir o número de questões `OK` (uma única marcação), `BRANCO` e `DUPLA_MARCACAO`.

#### 📌 Restrições Computacionais

* **Comparação estrita:** apenas valores maiores que $\mathrm{Th}$ são considerados marcações válidas; valores exatamente iguais ao limiar não devem ser contabilizados.
* **Letras das alternativas:** o índice $0$ corresponde à alternativa `A`, o índice $1$ à alternativa `B` e assim sucessivamente.
* **Múltiplas marcações:** sempre que duas ou mais alternativas excederem o limiar, a classificação deve ser `DUPLA_MARCACAO`, independentemente dos respectivos graus de preenchimento.

#### 🧠 Fundamentação Teórica

| Situação | Classificação | Interpretação |
|---|---|---|
| Exatamente uma alternativa acima do limiar | Letra da alternativa | Resposta válida |
| Nenhuma alternativa acima do limiar | `BRANCO` | Questão não respondida |
| Duas ou mais alternativas acima do limiar | `DUPLA_MARCACAO` | Resposta ambígua |

O limiar de preenchimento controla a sensibilidade do algoritmo. Valores muito baixos tendem a aumentar o número de `DUPLA_MARCACAO`, enquanto valores muito altos podem aumentar a quantidade de questões classificadas como `BRANCO`.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $Q$.
* Linha 2: Inteiro $K$.
* Linha 3: Número real $\mathrm{Th}$.
* Próximas $Q$ linhas: $K$ números reais, correspondentes aos graus de preenchimento das alternativas.
* Linha 1: Inteiro $Q$ e $K$.

**Saída:**

* $Q$ linhas, cada uma contendo a classificação da respectiva questão.
* Linha final: `OK: x  BRANCO: y  DUPLA_MARCACAO: z`.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 3<br>4<br>50<br>10 85 5 12<br>20 15 18 22<br>90 88 10 5 | B<br>BRANCO<br>DUPLA_MARCACAO<br>OK: 1  BRANCO: 1  DUPLA_MARCACAO: 1 | Na primeira questão apenas `B` supera o limiar; na segunda nenhuma alternativa o supera; na terceira, `A` e `B` excedem o limiar. |
| 1<br>2<br>50.0<br>50 50 | BRANCO<br>OK: 0  BRANCO: 1  DUPLA_MARCACAO: 0 | Valores iguais ao limiar não são considerados marcações válidas. |

In [3]:
#| label: fig-06-sim-ep03
#| fig-cap: "Simulador: Classificação de Marcações OMR"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim06_ep03" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim06_ep03 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim06_ep03 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim06_ep03 button:hover { background: #e8dfcf; }
  #sim06_ep03 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim06_ep03_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim06_ep03_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP06_03: Classificação de Marcações OMR</span>
  <span class="sim06_ep03_pill">4 Alternativas</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim06_ep03_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Limiar de Preenchimento (Th): <span id="sim06_ep03_vth" style="font-family:monospace; color:#26241d;">50</span>%
      </label>
    </div>
    
    <input id="sim06_ep03_th" type="range" min="0" max="100" step="1" value="50">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajuste o grau de preenchimento de cada bolha (A&ndash;D) e o limiar para observar a classificação resultante.
    </div>
  </div>

  <!-- Sliders das Bolhas (A-D) -->
  <div class="sim06_ep03_panel" style="margin-bottom:14px;">
    <div id="sim06_ep03_bubbles" style="display:grid; grid-template-columns:repeat(4, 1fr); gap:12px;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim06_ep03_debug" class="sim06_ep03_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep03(root){
    if (!root || root.dataset.sim06Ep03Init) return;
    root.dataset.sim06Ep03Init = "1";

    var letras = ['A', 'B', 'C', 'D'];
    var valores = [10, 85, 5, 12];
    var thEl  = root.querySelector('#sim06_ep03_th');
    var vthEl = root.querySelector('#sim06_ep03_vth');
    var box   = root.querySelector('#sim06_ep03_bubbles');
    var dbg   = root.querySelector('#sim06_ep03_debug');

    box.innerHTML = '';
    var sliders = [];

    letras.forEach(function(L, i){
      var col = document.createElement('div');
      col.style.cssText = 'text-align:center; background:#fafaf7; border:1px solid #e9e3d3; padding:10px; border-radius:8px;';
      col.innerHTML = '<div style="font-weight:700; font-size:12px; color:#5e5a4a; margin-bottom:6px;">' + L + '</div>' +
        '<input type="range" min="0" max="100" step="1" value="' + valores[i] + '" id="sim06_ep03_b' + i + '">' +
        '<div id="sim06_ep03_v' + i + '" style="font-family:monospace; font-weight:700; font-size:11px; color:#26241d; margin-top:6px;">' + valores[i] + '%</div>';
      box.appendChild(col);
      sliders.push(col.querySelector('#sim06_ep03_b' + i));
    });

    function render(){
      var th = parseFloat(thEl.value);
      vthEl.textContent = th.toFixed(0);
      var marcadas = [];

      sliders.forEach(function(s, i){
        var v = parseFloat(s.value);
        root.querySelector('#sim06_ep03_v' + i).textContent = v.toFixed(0) + '%';
        if (v > th) marcadas.push(letras[i]);
      });

      var resultado;
      if (marcadas.length === 0) {
        resultado = 'BRANCO';
        dbg.style.borderColor = '#e4dcc8';
        dbg.style.background  = '#fafaf7';
        dbg.style.color       = '#8a8371';
      } else if (marcadas.length === 1) {
        resultado = 'RESPOSTA: ' + marcadas[0];
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        resultado = 'DUPLA_MARCACAO (' + marcadas.join(', ') + ')';
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      }

      dbg.textContent = 'Classificação da questão: ' + resultado;
    }

    sliders.forEach(function(s){ s.addEventListener('input', render); });
    thEl.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep03(){
    var root = document.getElementById('sim06_ep03');
    if (root) initSim06Ep03(root); else setTimeout(tryInitSim06Ep03, 200);
  }
  tryInitSim06Ep03();
})();
</script>
""")

In [9]:
%%writefile EP06_03.py
# Código Python

Overwriting EP06_03.py


In [10]:
TestSuite("EP06_03.py").run()

### EP06_04 🟡 Estimador de Inclinação por Mediana Angular (*Deskew*)

Após a detecção de bordas e a aplicação da Transformada de Hough, obtém-se um conjunto de retas candidatas à orientação predominante do documento. Cada reta fornece uma estimativa do ângulo de inclinação, calculada por

$$
\text{ângulo} = \operatorname{rad2deg}(\theta) - 90.
$$

Entretanto, nem todas as retas correspondem às linhas do documento: algumas resultam de ruídos, sombras ou outros elementos da imagem. Neste exercício, você implementará a etapa de estimação robusta do ângulo de inclinação, filtrando os valores plausíveis e calculando sua mediana.

#### 📋 Diretrizes de Implementação

1. **Quantidade:** Ler o inteiro $M$, correspondente ao número de ângulos estimados.
2. **Ângulos:** Ler os $M$ valores reais, em graus.
3. **Filtragem:** Manter apenas os ângulos que satisfaçam **estritamente** $-45 < \text{ângulo} < 45$.
4. **Ausência de candidatos:** Se nenhum ângulo permanecer após a filtragem, imprimir exatamente `SEM_CORRECAO`.
5. **Mediana:** Caso existam ângulos válidos:
   - se a quantidade for ímpar, a mediana é o elemento central da sequência ordenada;
   - se for par, a mediana é a média aritmética dos dois elementos centrais.
6. **Saída:** Imprimir a mediana arredondada para duas casas decimais (arredondamento padrão, *round half away from zero*, , com `np.floor(img + 0.5)`).

#### 📌 Restrições Computacionais

* **Intervalo aberto:** ângulos iguais a $-45$ ou $45$ não devem ser considerados.
* **Precisão:** calcular a mediana utilizando os valores originais; o arredondamento deve ser realizado apenas na saída.
* **Caso vazio:** se não houver ângulos válidos, nenhuma mediana deve ser calculada.

#### 🧠 Fundamentação Teórica

| Situação | Resultado |
|---|---|
| Maioria dos ângulos concentrada em torno da inclinação real | A mediana aproxima a orientação do documento. |
| Poucos ângulos discrepantes (*outliers*) | A mediana sofre pouca influência desses valores. |
| Ângulos fora do intervalo $(-45^\circ,45^\circ)$ | São descartados antes do cálculo. |
| Nenhum ângulo válido | Não é aplicada correção (`SEM_CORRECAO`). |

A mediana é utilizada por ser mais robusta que a média na presença de poucos valores discrepantes, produzindo uma estimativa mais estável da inclinação predominante do documento.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $M$.
* Linha 2: $M$ números reais, correspondentes aos ângulos em graus.

**Saída:**

* Uma única linha contendo o ângulo estimado, com duas casas decimais, ou a palavra `SEM_CORRECAO` caso nenhum ângulo seja válido.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 5<br>-50 -10.5 2.3 2.3 47 | 2.30 | Apenas os ângulos no intervalo $(-45,45)$ são considerados; a mediana é $2{,}3$. |
| 4<br>-46 50 45 -45 | SEM_CORRECAO | Nenhum ângulo pertence ao intervalo aberto $(-45,45)$. |

In [4]:
#| label: fig-06-sim-ep04
#| fig-cap: "Simulador: Estimador de Inclinação por Mediana Angular"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim06_ep04" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim06_ep04 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim06_ep04 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim06_ep04 button:hover { background: #e8dfcf; }
  #sim06_ep04 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim06_ep04_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim06_ep04_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP06_04: Estimador de Inclinação por Mediana Angular (Deskew)</span>
  <span class="sim06_ep04_pill">mediana(-45&deg; &lt; &theta; &lt; 45&deg;)</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim06_ep04_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Ângulo do Ruído Extra (&theta;_ruido): <span id="sim06_ep04_vl" style="font-family:monospace; color:#26241d;">47</span>&deg;
      </label>
    </div>
    
    <input id="sim06_ep04_sl" type="range" min="-80" max="80" step="1" value="47">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Arraste o ângulo do ruído extra para dentro ou fora do intervalo [-45&deg;, +45&deg;] e veja como a mediana permanece estável.
    </div>
  </div>

  <!-- Exibição dos Ângulos Amostrados -->
  <div class="sim06_ep04_panel" style="margin-bottom:14px;">
    <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; text-align:center; letter-spacing:0.04em;">
      Amostras de Ângulos (Verde = Dentro da Faixa, Vermelho = Ruído Descartado)
    </div>
    <div id="sim06_ep04_pts" style="display:flex; gap:8px; flex-wrap:wrap; justify-content:center;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim06_ep04_debug" class="sim06_ep04_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep04(root){
    if (!root || root.dataset.sim06Ep04Init) return;
    root.dataset.sim06Ep04Init = "1";

    var base = [-10.5, 2.3, 2.3];
    var slEl  = root.querySelector('#sim06_ep04_sl');
    var vlEl  = root.querySelector('#sim06_ep04_vl');
    var ptsEl = root.querySelector('#sim06_ep04_pts');
    var dbg   = root.querySelector('#sim06_ep04_debug');

    function median(arr){
      var a = arr.slice().sort(function(x, y){ return x - y; });
      var n = a.length;
      if (n === 0) return null;
      var mid = Math.floor(n / 2);
      return (n % 2 === 1) ? a[mid] : (a[mid - 1] + a[mid]) / 2;
    }

    function render(){
      var extra = parseFloat(slEl.value);
      vlEl.textContent = extra;
      var todos = base.concat([extra, -50]);
      var validos = todos.filter(function(a){ return a > -45 && a < 45; });
      
      ptsEl.innerHTML = '';
      todos.forEach(function(a){
        var ok = a > -45 && a < 45;
        var div = document.createElement('div');
        div.style.cssText = 'padding:8px 12px; border-radius:8px; font-family:monospace; font-size:12px; font-weight:700; transition:all 0.15s ease;' +
          (ok ? 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;' : 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;');
        div.textContent = a + '°';
        ptsEl.appendChild(div);
      });

      var med = median(validos);

      if (med === null) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
        dbg.textContent = 'Válidos: []  |  Mediana estimada: SEM_CORRECAO';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
        dbg.textContent = 'Válidos: [' + validos.join(', ') + ']  |  Mediana estimada: ' + med.toFixed(2) + '°';
      }
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep04(){
    var root = document.getElementById('sim06_ep04');
    if (root) initSim06Ep04(root); else setTimeout(tryInitSim06Ep04, 200);
  }
  tryInitSim06Ep04();
})();
</script>
""")

In [12]:
%%writefile EP06_04.py
# Código Python

Overwriting EP06_04.py


In [13]:
TestSuite("EP06_04.py").run()

### EP06_05 🟠 Normalização de Fundo por Divisão (Correção de Iluminação)

Um formulário foi fotografado sob iluminação não uniforme, fazendo com que um lado da folha apareça mais claro que o outro. Nessas condições, a limiarização global por Otsu pode produzir resultados insatisfatórios, pois um único limiar não separa adequadamente texto e fundo em toda a imagem. A solução apresentada no capítulo consiste em **normalizar o fundo**, dividindo a imagem original por uma versão fortemente suavizada de si mesma, que representa a iluminação de baixa frequência.

Neste exercício, a imagem original e o fundo suavizado (equivalente ao resultado de um `cv2.GaussianBlur` com $\sigma$ elevado) já são fornecidos. Sua tarefa é implementar a etapa de normalização que produz a imagem corrigida.

#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Imagem original:** Ler os $L \times C$ valores inteiros da matriz `img` (intensidades entre 0 e 255).
3. **Fundo estimado:** Ler os $L \times C$ valores inteiros da matriz `bg` (intensidades entre 0 e 255, sempre estritamente maiores que zero).
4. **Normalização:** Para cada posição $(i,j)$, calcular
$$
\text{valor}(i,j)=
\frac{\text{img}(i,j)}{\text{bg}(i,j)}\times255.
$$
5. **Arredondamento:** Arredondar o resultado para o inteiro mais próximo (*round half away from zero*, com `np.floor(img + 0.5)`).
6. **Saturação:** Limitar o valor obtido ao intervalo $[0,255]$.
7. **Saída:** Imprimir a matriz `img_norm` resultante.

#### 📌 Restrições Computacionais

* **Divisão por zero:** a entrada garante $\text{bg}(i,j)>0$ em todas as posições.
* **Ordem das operações:** primeiro arredondar, depois aplicar a saturação.
* **Processamento independente:** cada pixel deve ser normalizado individualmente, sem utilizar informações dos pixels vizinhos.

#### 🧠 Fundamentação Teórica

| Situação | Efeito da normalização |
|---|---|
| $\text{img}(i,j)=\text{bg}(i,j)$ | Resultado igual a $255$, correspondente ao fundo normalizado. |
| $\text{img}(i,j)<\text{bg}(i,j)$ | Resultado menor que $255$, preservando regiões mais escuras, como texto. |
| $\text{img}(i,j)>\text{bg}(i,j)$ | Resultado superior a $255$, posteriormente saturado. |
| Fundo com iluminação não uniforme | A divisão reduz as variações lentas de iluminação, tornando a imagem mais homogênea. |

A divisão pelo fundo estimado reduz os efeitos da iluminação não uniforme e preserva o contraste entre o primeiro plano e o fundo, facilitando as etapas posteriores de segmentação.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Próximas $L$ linhas: elementos da matriz `img`.
* Próximas $L$ linhas: elementos da matriz `bg`.

**Saída:**

* Matriz `img_norm`, com $L$ linhas e $C$ colunas, contendo valores inteiros separados por espaço.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 2<br>2<br>60 120<br>180 40<br>100 100<br>200 80 | 153 255<br>230 128 | Valores superiores a $255$ devem ser saturados; $180/200\times255=229{,}5$ resulta em $230$ após o arredondamento. |
| 1<br>3<br>30 60 90<br>60 60 60 | 128 255 255 | Apenas o primeiro valor permanece abaixo de $255$ após a normalização. |

In [5]:
#| label: fig-06-sim-ep05
#| fig-cap: "Simulador: Normalização de Fundo por Divisão"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim06_ep05" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim06_ep05 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim06_ep05 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim06_ep05 button:hover { background: #e8dfcf; }
  #sim06_ep05 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim06_ep05_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim06_ep05_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim05_ep05_cell { width: 40px; height: 40px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 9px; font-weight: 700; font-family: monospace; border: 1px solid #e4dcc8; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP06_05: Normalização de Fundo por Divisão</span>
  <span class="sim06_ep05_pill">(img / bg) &times; 255</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim06_ep05_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Intensidade do Fundo à Esquerda (bg_esq): <span id="sim06_ep05_vl" style="font-family:monospace; color:#26241d;">100</span>
      </label>
    </div>
    
    <input id="sim06_ep05_sl" type="range" min="40" max="220" step="5" value="100">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajuste o gradiente de fundo (esquerda &rarr; direita) e observe como a divisão cancela a variação de iluminação.
    </div>
  </div>

  <!-- Exibição das Matrizes 1x4 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(160px, 1fr)); gap:12px; margin-bottom:14px;">
    
    <div class="sim06_ep05_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        img (Original)
      </div>
      <div id="sim06_ep05_g_img" style="display:grid; grid-template-columns:repeat(4, 40px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim06_ep05_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        bg (Fundo Suavizado)
      </div>
      <div id="sim06_ep05_g_bg" style="display:grid; grid-template-columns:repeat(4, 40px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim06_ep05_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        img_norm (Saída)
      </div>
      <div id="sim06_ep05_g_out" style="display:grid; grid-template-columns:repeat(4, 40px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim06_ep05_debug" class="sim06_ep05_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep05(root){
    if (!root || root.dataset.sim06Ep05Init) return;
    root.dataset.sim06Ep05Init = "1";

    var linha_img = [90, 90, 90, 90];
    var slEl = root.querySelector('#sim06_ep05_sl');
    var vlEl = root.querySelector('#sim06_ep05_vl');
    var gImg = root.querySelector('#sim06_ep05_g_img');
    var gBg  = root.querySelector('#sim06_ep05_g_bg');
    var gOut = root.querySelector('#sim06_ep05_g_out');
    var dbg  = root.querySelector('#sim06_ep05_debug');

    function roundHalfAway(x){
      return x >= 0 ? Math.floor(x + 0.5) : Math.ceil(x - 0.5);
    }

    function cellStyle(v){
      var g = Math.max(0, Math.min(255, v));
      return 'background:rgb(' + g + ',' + g + ',' + g + '); color:' + (g > 140 ? '#000000' : '#ffffff') + ';';
    }

    function render(){
      var bgEsq = parseInt(slEl.value, 10);
      vlEl.textContent = bgEsq;

      // Gradiente linear de bgEsq até 200 na direita, 4 colunas
      var bg = [];
      for (var j = 0; j < 4; j++){
        bg.push(Math.round(bgEsq + (200 - bgEsq) * j / 3));
      }

      gImg.innerHTML = '';
      gBg.innerHTML  = '';
      gOut.innerHTML = '';
      
      var out = [];
      for (var j = 0; j < 4; j++){
        var v = (linha_img[j] / bg[j]) * 255;
        var r = roundHalfAway(v);
        var sat = Math.max(0, Math.min(255, r));
        out.push(sat);

        var ci = document.createElement('div');
        ci.className = 'sim05_ep05_cell';
        ci.style.cssText = cellStyle(linha_img[j]);
        ci.textContent = linha_img[j];
        gImg.appendChild(ci);

        var cb = document.createElement('div');
        cb.className = 'sim05_ep05_cell';
        cb.style.cssText = cellStyle(bg[j]);
        cb.textContent = bg[j];
        gBg.appendChild(cb);

        var co = document.createElement('div');
        co.className = 'sim05_ep05_cell';
        co.style.cssText = cellStyle(sat);
        co.textContent = sat;
        gOut.appendChild(co);
      }

      dbg.textContent = 'bg = [' + bg.join(', ') + ']  |  img_norm = [' + out.join(', ') + ']';
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep05(){
    var root = document.getElementById('sim06_ep05');
    if (root) initSim06Ep05(root); else setTimeout(tryInitSim06Ep05, 200);
  }
  tryInitSim06Ep05();
})();
</script>
""")

In [15]:
%%writefile EP06_05.py
# Código Python

Overwriting EP06_05.py


In [16]:
TestSuite("EP06_05.py").run()

### EP06_06 🔴 Mapa de Variância Local para Detecção de Textura

Uma fábrica de tecidos precisa inspecionar rolos de pano em tempo real, sem dispor de uma imagem de referência — cada rolo apresenta pequenas variações naturais. Nessa situação, a estratégia apresentada no capítulo consiste em analisar a **homogeneidade local da textura**: regiões uniformes apresentam baixa variância de intensidade em pequenas vizinhanças, enquanto riscos, manchas e falhas de fabricação produzem aumentos locais dessa variância.

Neste exercício, você implementará o núcleo desse método, calculando a variância local em uma janela deslizante e gerando uma máscara binária que identifica as regiões cuja variância excede um limiar.

#### 📋 Diretrizes de Implementação

1. **Dimensões e parâmetros:** Ler os inteiros $L$, $C$, $k$ (tamanho da janela, sempre ímpar) e $T$ (limiar de variância).
2. **Imagem:** Ler os $L \times C$ valores inteiros da matriz de textura (intensidades entre 0 e 255).
3. **Tratamento das bordas:** Quando a janela ultrapassar os limites da imagem, utilizar **replicação de borda**, isto é, repetir o valor do pixel válido mais próximo.
4. **Média local:** Para cada posição $(i,j)$, calcular
$$
\mu(i,j)=
\frac{1}{k^2}
\sum_{(p,q)\in\text{janela}}
\text{textura}(p,q).
$$
5. **Variância local:** Calcular a variância populacional da janela,
$$
\sigma^2(i,j)=
\frac{1}{k^2}
\sum_{(p,q)\in\text{janela}}
\left(\text{textura}(p,q)-\mu(i,j)\right)^2,
$$
ou, de forma equivalente,
$$
\sigma^2(i,j)=\overline{x^2}-\mu(i,j)^2,
$$
em que $\overline{x^2}$ representa a média dos quadrados das intensidades.

6. **Arredondamento:** Arredondar a variância para o inteiro mais próximo (*round half away from zero*, com `np.floor(res_norm + 0.5)`).

7. **Limiarização:** Definir $\text{máscara}(i,j)=1$ se a variância arredondada for **estritamente maior** que $T$; caso contrário, definir $\text{máscara}(i,j)=0$.

8. **Saída:** Imprimir a máscara binária resultante.

#### 📌 Restrições Computacionais

* **Replicação de borda:** utilizar o valor do pixel válido mais próximo sempre que a janela ultrapassar os limites da imagem.
* **Variância populacional:** utilizar denominador $k^2$, nunca $k^2-1$.
* **Comparação estrita:** a máscara deve ser calculada utilizando a condição $\sigma^2_{\text{arred}}>T$.
* **Janela ímpar:** o valor de $k$ é sempre ímpar, garantindo um pixel central.

#### 🧠 Fundamentação Teórica

| Situação | Variância local | Interpretação |
|---|---|---|
| Região uniforme | Baixa | Intensidades semelhantes na vizinhança. |
| Região contendo defeito | Alta | A presença de intensidades distintas aumenta a dispersão dos valores. |
| Janela pequena | Maior sensibilidade a detalhes e ruído | Detecta alterações localizadas. |
| Janela grande | Resposta mais suave | Evidencia defeitos maiores, porém reduz a precisão de sua localização. |

A variância local mede a dispersão das intensidades em uma vizinhança. Regiões homogêneas apresentam baixa variância, enquanto alterações na textura aumentam essa medida, permitindo identificar possíveis defeitos por meio de uma simples limiarização.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $k$ (ímpar).
* Linha 4: Inteiro $T$.
* Próximas $L$ linhas: elementos inteiros da matriz de textura.

**Saída:**

* Máscara binária (valores 0 ou 1), com $L$ linhas e $C$ colunas.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 3<br>3<br>3<br>50<br>10 10 10<br>10 10 10<br>10 90 10 | 0 0 0<br>1 1 1<br>1 1 1 | O defeito aumenta a variância em todas as janelas que o contêm. |
| 2<br>2<br>3<br>5<br>100 100<br>100 100 | 0 0<br>0 0 | A textura é uniforme; a variância é nula em toda a imagem. |

In [6]:
#| label: fig-06-sim-ep06
#| fig-cap: "Simulador: Mapa de Variância Local para Detecção de Textura"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim06_ep06" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim06_ep06 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim06_ep06 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim06_ep06 button:hover { background: #e8dfcf; }
  #sim06_ep06 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim06_ep06_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim06_ep06_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim06_ep06_cell { width: 44px; height: 44px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; border: 1px solid #e4dcc8; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP06_06: Variância Local (Detecção de Textura)</span>
  <span class="sim06_ep06_pill">&sigma;&sup2; = m&eacute;dia(x&sup2;) &minus; m&eacute;dia(x)&sup2;</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim06_ep06_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Intensidade do Defeito (Posição Central): <span id="sim06_ep06_vl_def" style="font-family:monospace; color:#26241d;">90</span>
      </label>
    </div>
    <input id="sim06_ep06_sl_def" type="range" min="10" max="255" step="5" value="90">

    <div style="display:flex; justify-content:space-between; align-items:center; margin:10px 0 4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Limiar (T): <span id="sim06_ep06_vl_t" style="font-family:monospace; color:#26241d;">50</span>
      </label>
    </div>
    <input id="sim06_ep06_sl_t" type="range" min="0" max="2000" step="10" value="50">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajuste o valor do defeito e o limiar T; observe como a janela 3&times;3 espalha a detecção pela vizinhança.
    </div>
  </div>

  <!-- Exibição das Grades 3x3 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim06_ep06_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Textura (3&times;3)
      </div>
      <div id="sim06_ep06_g_tex" style="display:grid; grid-template-columns:repeat(3, 44px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim06_ep06_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Máscara de Defeito
      </div>
      <div id="sim06_ep06_g_mask" style="display:grid; grid-template-columns:repeat(3, 44px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim06_ep06_debug" class="sim06_ep06_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep06(root){
    if (!root || root.dataset.sim06Ep06Init) return;
    root.dataset.sim06Ep06Init = "1";

    var slDef = root.querySelector('#sim06_ep06_sl_def');
    var vlDef = root.querySelector('#sim06_ep06_vl_def');
    var slT   = root.querySelector('#sim06_ep06_sl_t');
    var vlT   = root.querySelector('#sim06_ep06_vl_t');
    var gTex  = root.querySelector('#sim06_ep06_g_tex');
    var gMask = root.querySelector('#sim06_ep06_g_mask');
    var dbg   = root.querySelector('#sim06_ep06_debug');

    function roundHalfAway(x){
      return x >= 0 ? Math.floor(x + 0.5) : Math.ceil(x - 0.5);
    }

    function clampIdx(v, n){
      return Math.max(0, Math.min(n - 1, v));
    }

    function render(){
      var defeito = parseInt(slDef.value, 10);
      var T       = parseInt(slT.value, 10);
      vlDef.textContent = defeito;
      vlT.textContent   = T;

      var N = 3;
      var tex = [[10, 10, 10], [10, defeito, 10], [10, 10, 10]];

      gTex.innerHTML  = '';
      gMask.innerHTML = '';
      
      var mask = [];
      for (var i = 0; i < N; i++){
        var row = [];
        for (var j = 0; j < N; j++){
          var vals = [];
          for (var di = -1; di <= 1; di++){
            for (var dj = -1; dj <= 1; dj++){
              var pi = clampIdx(i + di, N);
              var pj = clampIdx(j + dj, N);
              vals.push(tex[pi][pj]);
            }
          }
          var mean = vals.reduce(function(a, b){ return a + b; }, 0) / vals.length;
          var meanSq = vals.reduce(function(a, b){ return a + b * b; }, 0) / vals.length;
          var varr = meanSq - mean * mean;
          var varRound = roundHalfAway(varr);
          row.push(varRound > T ? 1 : 0);
        }
        mask.push(row);
      }

      var total = 0;
      for (var i = 0; i < N; i++){
        for (var j = 0; j < N; j++){
          var g = tex[i][j];
          var ct = document.createElement('div');
          ct.className = 'sim06_ep06_cell';
          ct.style.cssText = 'background:rgb(' + g + ',' + g + ',' + g + '); color:' + (g > 140 ? '#000000' : '#ffffff') + ';';
          ct.textContent = g;
          gTex.appendChild(ct);

          var m = mask[i][j];
          if (m) total++;

          var cm = document.createElement('div');
          cm.className = 'sim06_ep06_cell';
          if (m) {
            cm.style.cssText = 'background:#fdecea; color:#c0392b; border:1px solid #f5b7b1;';
          } else {
            cm.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
          }
          cm.textContent = m;
          gMask.appendChild(cm);
        }
      }

      if (total > 0) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      }

      dbg.textContent = 'defeito = ' + defeito + '  |  T = ' + T + '  |  Pixels marcados: ' + total + ' / 9';
    }

    slDef.addEventListener('input', render);
    slT.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep06(){
    var root = document.getElementById('sim06_ep06');
    if (root) initSim06Ep06(root); else setTimeout(tryInitSim06Ep06, 200);
  }
  tryInitSim06Ep06();
})();
</script>
""")

In [18]:
%%writefile EP06_06.py
# Código Python

Overwriting EP06_06.py


In [19]:
TestSuite("EP06_06.py").run()

### EP06_07 🟣 *Pipeline* de Inspeção Industrial: Registro por Translação e Subtração

Em uma linha de produção, uma câmera fixa fotografa cada peça que passa pela esteira, comparando-a a uma imagem de referência sem defeitos. O problema: pequenas vibrações da esteira deslocam a peça em relação à posição de referência a cada captura. Se a subtração de imagens for aplicada diretamente, sem correção, o deslocamento por si só já gera diferenças enormes — **falsos positivos** que mascaram os defeitos reais.

Este é o exercício mais completo do capítulo: você deve **primeiro registrar** (alinhar geometricamente) a imagem capturada usando um deslocamento conhecido $(dx, dy)$, fornecido por um sensor de posição da esteira, e **só então aplicar a subtração** com limiarização, exatamente como descrito na seção de inspeção industrial.

#### 📋 Diretrizes de Implementação

1. **Dimensões e parâmetros:** Ler $L$, $C$ (dimensões das imagens), o deslocamento inteiro conhecido $dx, dy$ (podendo ser negativos) e o limiar de detecção $T$ (inteiro).
2. **Imagens:** Ler a matriz de referência (`ref`, $L\times C$, sem defeitos) e a matriz capturada (`cap`, $L\times C$, possivelmente deslocada e com defeito).
3. **Registro por translação:** Construir a imagem alinhada `alin` aplicando o deslocamento $(dx,dy)$ recebido:
$$
\text{alin}(i,j) = \begin{cases} \text{cap}(i+dy,\; j+dx), & \text{se } (i+dy,\ j+dx) \in [0,L)\times[0,C) \\ 0, & \text{caso contrário} \end{cases}
$$
4. **Preenchimento de borda:** As posições que "saem" da imagem capturada após o deslocamento recebem o valor **0** (*zero-padding* — fora do campo de visão da câmera; **note que este exercício usa zero, diferente da replicação de borda do EP06_06**).
5. **Diferença absoluta:** Calcular, pixel a pixel,
$$
\text{diff}(i,j) = |\text{ref}(i,j) - \text{alin}(i,j)|
$$
6. **Limiarização:** Definir $\text{máscara}(i,j) = 1$ se $\text{diff}(i,j) > T$; caso contrário, $\text{máscara}(i,j) = 0$.
7. **Saída:** Nesta ordem — (a) a matriz `alin` ($L\times C$); (b) a máscara de defeito ($L\times C$); (c) uma última linha com o total de pixels classificados como defeituosos.

#### 📌 Restrições Computacionais

* ***Zero-padding*, não replicação:** posições fora dos limites da imagem capturada, após o deslocamento, valem exatamente 0 — este é o ponto que mais diferencia este exercício do EP06_06.
* **Comparação estrita:** $\text{diff}(i,j) > T$.
* **Sinal de $(dx,dy)$:** o deslocamento pode ser positivo ou negativo; a fórmula do passo 3 deve ser aplicada literalmente, sem inverter os sinais.
* **Todos os valores são inteiros:** não há arredondamento nesta etapa.

#### 🧠 Fundamentação Teórica

| Etapa omitida | Consequência |
|---|---|
| Pular o registro geométrico | A borda inteira da imagem (introduzida pelo deslocamento) é marcada como "defeito" — falso positivo sistemático |
| Registro com $(dx,dy)$ incorreto | Peça e referência ficam desalinhadas; a subtração detecta contornos deslocados, não defeitos reais |
| Limiar $T$ muito baixo | Ruído de captura (variações de 1–2 níveis de cinza) é confundido com defeito |
| Limiar $T$ muito alto | Defeitos sutis deixam de ser detectados |

O registro geométrico e a subtração são etapas complementares: o primeiro garante que ambas as imagens representem exatamente a mesma cena no mesmo referencial espacial; o segundo isola o que realmente mudou entre elas — idealmente, apenas os defeitos.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Dois inteiros $dx$ e $dy$, separados por espaço.
* Linha 4: Inteiro $T$.
* Próximas $L$ linhas: elementos inteiros da matriz `ref`.
* Próximas $L$ linhas: elementos inteiros da matriz `cap`.

**Saída:**

* $L$ linhas com a matriz `alin`.
* $L$ linhas com a máscara de defeito (0/1).
* Última linha: `Total de pixels defeituosos: X`.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 3<br>3<br>1 0<br>30<br>50 50 50<br>50 50 50<br>50 50 50<br>0 50 50<br>0 50 90<br>0 50 50 | 50 50 0<br>50 90 0<br>50 50 0<br>0 0 1<br>0 1 1<br>0 0 1<br>Total de pixels defeituosos: 4 | $dx=1$ desloca a leitura uma coluna à direita; a última coluna de `alin` fica sem correspondência <br> (vira 0) e é sistematicamente marcada; o defeito real (90) também é detectado. |
| 2<br>2<br>0 0<br>20<br>10 10<br>10 10<br>10 10<br>10 60 | 10 10<br>10 60<br>0 0<br>0 1<br>Total de pixels defeituosos: 1 | Sem deslocamento ($dx=dy=0$): `alin` é idêntica a `cap`; apenas o defeito real (60) é detectado. |


In [7]:
#| label: fig-06-sim-ep07
#| fig-cap: "Simulador: Pipeline de Inspeção — Registro por Translação e Subtração"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim06_ep07" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim06_ep07 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim06_ep07 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim06_ep07 button:hover { background: #e8dfcf; }
  #sim06_ep07 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim06_ep07_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim06_ep07_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim06_ep07_cell { width: 40px; height: 40px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 10px; font-weight: 700; font-family: monospace; border: 1px solid #e4dcc8; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP06_07: Registro por Translação + Subtração</span>
  <span class="sim06_ep07_pill">|ref &minus; alin(dx,dy)| &gt; T</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim06_ep07_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Deslocamento Horizontal (dx): <span id="sim06_ep07_vl_dx" style="font-family:monospace; color:#26241d;">1</span>
      </label>
    </div>
    <input id="sim06_ep07_sl_dx" type="range" min="-2" max="2" step="1" value="1">

    <div style="display:flex; justify-content:space-between; align-items:center; margin:10px 0 4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Limiar (T): <span id="sim06_ep07_vl_t" style="font-family:monospace; color:#26241d;">30</span>
      </label>
    </div>
    <input id="sim06_ep07_sl_t" type="range" min="0" max="100" step="5" value="30">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajuste o deslocamento da esteira (dx) e o limiar T. Observe como a borda "fantasma" desaparece quando dx = 0.
    </div>
  </div>

  <!-- Exibição das Grades 3x3 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(160px, 1fr)); gap:12px; margin-bottom:14px;">
    
    <div class="sim06_ep07_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        ref
      </div>
      <div id="sim06_ep07_g_ref" style="display:grid; grid-template-columns:repeat(3, 40px); gap:3px; justify-content:center;"></div>
    </div>

    <div class="sim06_ep07_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        alin (registrada)
      </div>
      <div id="sim06_ep07_g_alin" style="display:grid; grid-template-columns:repeat(3, 40px); gap:3px; justify-content:center;"></div>
    </div>

    <div class="sim06_ep07_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        máscara
      </div>
      <div id="sim06_ep07_g_mask" style="display:grid; grid-template-columns:repeat(3, 40px); gap:3px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim06_ep07_debug" class="sim06_ep07_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep07(root){
    if (!root || root.dataset.sim06Ep07Init) return;
    root.dataset.sim06Ep07Init = "1";

    var N = 3;
    var ref = [[50, 50, 50], [50, 50, 50], [50, 50, 50]];
    // cap representa a peça deslocada 1 px à direita (col 0 = 0) mais um defeito em (1,2)
    var cap = [[0, 50, 50], [0, 50, 90], [0, 50, 50]];

    var slDx  = root.querySelector('#sim06_ep07_sl_dx');
    var vlDx  = root.querySelector('#sim06_ep07_vl_dx');
    var slT   = root.querySelector('#sim06_ep07_sl_t');
    var vlT   = root.querySelector('#sim06_ep07_vl_t');
    var gRef  = root.querySelector('#sim06_ep07_g_ref');
    var gAlin = root.querySelector('#sim06_ep07_g_alin');
    var gMask = root.querySelector('#sim06_ep07_g_mask');
    var dbg   = root.querySelector('#sim06_ep07_debug');

    function cellStyle(g){
      var v = Math.max(0, Math.min(255, g));
      return 'background:rgb(' + v + ',' + v + ',' + v + '); color:' + (v > 140 ? '#000000' : '#ffffff') + ';';
    }

    function render(){
      var dx = parseInt(slDx.value, 10);
      var T  = parseInt(slT.value, 10);
      vlDx.textContent = dx;
      vlT.textContent  = T;

      gRef.innerHTML  = '';
      gAlin.innerHTML = '';
      gMask.innerHTML = '';

      var alin = [], mask = [], total = 0;

      for (var i = 0; i < N; i++){
        var rowA = [], rowM = [];
        for (var j = 0; j < N; j++){
          var pj = j + dx;
          var v = (pj >= 0 && pj < N) ? cap[i][pj] : 0;
          rowA.push(v);

          var diff = Math.abs(ref[i][j] - v);
          var m = diff > T ? 1 : 0;
          if (m) total++;
          rowM.push(m);
        }
        alin.push(rowA);
        mask.push(rowM);
      }

      for (var i = 0; i < N; i++){
        for (var j = 0; j < N; j++){
          var cr = document.createElement('div');
          cr.className = 'sim06_ep07_cell';
          cr.style.cssText = cellStyle(ref[i][j]);
          cr.textContent = ref[i][j];
          gRef.appendChild(cr);

          var ca = document.createElement('div');
          ca.className = 'sim06_ep07_cell';
          ca.style.cssText = cellStyle(alin[i][j]);
          ca.textContent = alin[i][j];
          gAlin.appendChild(ca);

          var m = mask[i][j];
          var cm = document.createElement('div');
          cm.className = 'sim06_ep07_cell';
          if (m) {
            cm.style.cssText = 'background:#fdecea; color:#c0392b; border:1px solid #f5b7b1;';
          } else {
            cm.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
          }
          cm.textContent = m;
          gMask.appendChild(cm);
        }
      }

      if (total > 0) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      }

      dbg.textContent = 'dx = ' + dx + '  |  T = ' + T + '  |  Total de pixels defeituosos: ' + total + ' / 9';
    }

    slDx.addEventListener('input', render);
    slT.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep07(){
    var root = document.getElementById('sim06_ep07');
    if (root) initSim06Ep07(root); else setTimeout(tryInitSim06Ep07, 200);
  }
  tryInitSim06Ep07();
})();
</script>
""")

In [21]:
%%writefile EP06_07.py
# Código Python

Overwriting EP06_07.py


In [22]:
TestSuite("EP06_07.py").run()

### EP06_08 ⚫ Segmentação e Decodificação Real de *QRCode* com OpenCV

Nos exercícios anteriores, as grandezas intermediárias do *pipeline* de processamento de imagens — como áreas, perímetros, variâncias e deslocamentos — foram fornecidas diretamente ou calculadas a partir de matrizes numéricas, sem a necessidade de bibliotecas especializadas de Visão Computacional. Neste exercício de encerramento do capítulo, essa restrição é removida de forma intencional: será utilizada a biblioteca **OpenCV** (`cv2`) para localizar e decodificar um *QRCode* real presente em uma cena.

A proposta reproduz um fluxo simplificado de sistemas empregados em inspeção visual, automação industrial e leitura automática de documentos. Para manter a entrada de dados acessível ao contexto educacional, o carregamento da imagem será integrado à biblioteca didática `morph`, por meio da função `mm.readImg`.

A cena é fornecida no formato **PGM ASCII (P2)** e contém um único *QRCode* válido, além de diversos **objetos distratores**, como retângulos, regiões de ruído texturizado e blocos isolados. A segmentação baseada apenas em propriedades geométricas — como área e formato aproximadamente quadrado — é necessária para reduzir o espaço de busca, mas não é suficiente para identificar o código correto. A confirmação final será realizada exclusivamente pela tentativa de decodificação utilizando `cv2.QRCodeDetector`, procedimento compatível com aplicações reais de reconhecimento automático.

#### 📋 Diretrizes de Implementação

1. **Leitura das dimensões e parâmetros**

   Ler, nesta ordem, por meio da entrada padrão:

   - uma linha contendo o número de linhas $L$;
   - uma linha contendo o número de colunas $C$;
   - uma linha contendo os quatro parâmetros do algoritmo separados por espaço:
     - limiar de binarização $T$ (inteiro);
     - área mínima $A_{\text{min}}$ (inteiro);
     - tolerância de aspecto $\text{tol}$ (real);
     - margem $M$ (inteiro, em pixels).

2. **Carregamento da imagem**

   Utilizar a função didática `f = mm.readImg(L, C)` para ler os $L \times C$ valores da imagem em tons de cinza, obtendo um *array* NumPy do tipo `uint8`.

3. **Binarização**

   Aplicar limiarização binária invertida utilizando o limiar $T$. Todo pixel da imagem original com intensidade estritamente maior que $T$ deve ser convertido para 255, enquanto os demais devem assumir o valor 0.

4. **Detecção de contornos**

   Extrair os componentes conectados externos utilizando `cv2.findContours(...)` com os parâmetros:

   * `cv2.RETR_EXTERNAL`;
   * `cv2.CHAIN_APPROX_SIMPLE`.

5. **Filtragem geométrica**

   Para cada contorno encontrado:

   * calcular o retângulo delimitador `(x, y, w, h)` por meio de `cv2.boundingRect`;
   * manter apenas os candidatos que satisfaçam simultaneamente:

     **Área mínima**

     $$
     w \times h > A_{\text{min}}
     $$

     **Razão de aspecto**

     $$
     \left|\frac{w}{h}-1\right| \le \text{tol}
     $$

6. **Ordenação dos candidatos**

   Ordenar os candidatos pela área do retângulo delimitador

   $$
   w \times h
   $$

   em ordem decrescente.

   Em caso de empate, preservar a ordem originalmente retornada por `cv2.findContours`.

7. **Verificação por decodificação**

   Para cada candidato, seguindo a ordem estabelecida:

   * expandir o retângulo em $M$ pixels nas quatro direções;
   * limitar os índices para permanecerem dentro da imagem;
   * extrair o recorte diretamente da imagem original `f`;
   * aplicar `cv2.QRCodeDetector().detectAndDecode(...)` sobre esse recorte.

8. **Critério de parada**

   Interromper imediatamente o processamento quando o primeiro candidato produzir uma *string* decodificada não vazia.

9. **Caso não encontrado**

   Se nenhum candidato for decodificado com sucesso, imprimir exatamente: `QRCODE_NAO_ENCONTRADO`

10. **Saída (caso encontrado)**

    Imprimir duas linhas.

    Primeira linha: `linha coluna altura largura` utilizando o retângulo delimitador **original**, antes da expansão pela margem $M$.

    Segunda linha: `texto_decodificado`


#### 📌 Restrições Computacionais

* Utilizar funções do OpenCV para realizar a binarização, a detecção de contornos, o cálculo do retângulo delimitador e a decodificação do QRCode.
* A filtragem geométrica deve ocorrer obrigatoriamente antes da etapa de decodificação.
* Utilizar exclusivamente o limiar fixo $T$ fornecido na entrada. Não é permitido utilizar métodos automáticos de limiarização, como Otsu ou limiarização adaptativa.
* Garantir que os recortes enviados ao decodificador permaneçam dentro dos limites da imagem.


#### 🧠 Fundamentação Teórica

| Etapa                    | Papel no pipeline                                                                                      | Consequência se omitida                                                                    |
| ------------------------ | ------------------------------------------------------------------------------------------------------ | ------------------------------------------------------------------------------------------ |
| **Filtragem geométrica** | Reduz o espaço de busca selecionando apenas regiões compatíveis com a geometria esperada de um QRCode. | O decodificador processaria todos os contornos, incluindo ruídos e objetos distratores.    |
| **Decodificação**        | Confirma semanticamente se o candidato contém um QRCode válido.                                        | Objetos geometricamente semelhantes poderiam ser classificados incorretamente como QRCode. |
| **Margem $M$**           | Preserva a *quiet zone* ao redor do código, facilitando sua detecção.                                  | A ausência dessa margem pode impedir o alinhamento e a leitura correta do código.          |

Este exercício integra conceitos estudados ao longo do capítulo em um único *pipeline* de Visão Computacional. A segmentação reduz o conjunto de regiões candidatas por meio de características geométricas, enquanto a etapa de decodificação valida o conteúdo da região utilizando um algoritmo especializado de reconhecimento.


#### 📦 Especificação de Entrada e Saída (VPL)

##### Estrutura de Entrada

```
L
C
T A_min tol M
[matriz da imagem]
```

##### Estrutura de Saída (Sucesso)

```
linha coluna altura largura
texto_decodificado
```

##### Estrutura de Saída (Falha)

```
QRCODE_NAO_ENCONTRADO
```

#### 📌 Arquivos de Referência (.pgm)


Para fins de validação, depuração local e análise de matrizes reais de pixels, os arquivos de imagem gerados no padrão ASCII P2 encontram-se disponíveis no diretório do projeto. Você pode utilizá-los para testar em decodificados do seu celular a aderência do seu código (salvar *.pgm localmente para visualizar): 

* 📥 **[Caso 1: Padrão Normal](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso1_Normal.pgm)** – Contém um único código perfeitamente centralizado com distratores geométricos simples na periferia. 
* 📥 **[Caso 2: Cenário Complexo](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso2_Complexo.pgm)** – Apresenta maior densidade de ruído texturizado e múltiplos distratores candidatos que testam os limites da filtragem por aspecto. 
* 📥 **[Caso 3: Mensagem Expandida](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso3_MensagemSecreta.pgm)** – Contém um QRCode estruturado a partir de uma cadeia de caracteres de maior comprimento, gerando maior densidade de módulos internos. 
* 📥 **[Caso 4: Geometria Compacta](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso4_Excelente.pgm)** – Avalia o comportamento do pipeline sob condições otimizadas de contraste e posicionamento limítrofe. 
* 📥 **[Caso 5: Cenário de Exclusão](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso5_Nao_Encontrado.pgm)** – Imagem composta puramente por elementos distratores de alta área, projetada para validar o comportamento de falha controlada do programa.

In [15]:
#| label: fig-06-sim-ep08
#| fig-cap: "Simulador: Segmentação Geométrica + Verificação por Decodificação de QRCode"
#| echo: false
#| output: true

from IPython.display import HTML
HTML("""
<div id="sim06_ep08" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<!-- Cabeçalho no padrão institucional -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">📋 Simulador EP06_08: Segmentação e Decodificação de QRCode</span>
  <span style="font-size:10px;font-weight:700;padding:3px 10px;border-radius:40px;border:1px solid #e4dcc8;background:#26241d;color:#7ee7c6;font-family:monospace;">Filtro Geométrico &rarr; Parada Semântica</span>
</div>

<div style="padding:16px;background:#ffffff;">
  <p style="margin:0 0 14px 0;font-size:11px;color:#8a8371;line-height:1.5;text-align:center;font-weight:600;">
    Ajuste interativamente os parâmetros de entrada do algoritmo (A_min e tol) para verificar quais componentes são filtrados geometricamente e como o critério de parada por análise semântica interrompe a varredura da fila.
  </p>
  
  <div style="display:flex;gap:14px;margin-bottom:14px;flex-wrap:wrap;">
    <!-- Slider Area Minima -->
    <div style="flex:1;min-width:200px;background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#5e5a4a;">Área mínima (A_min, px&sup2;)</label>
        <span id="sim06_ep08_vl_amin" style="font-family:monospace;font-weight:700;color:#26241d;">250</span>
      </div>
      <input id="sim06_ep08_sl_amin" style="width:100%;accent-color:#26241d;cursor:pointer;height:4px;" max="3000" min="0" step="50" type="range" value="250">
    </div>
    
    <!-- Slider Tolerancia -->
    <div style="flex:1;min-width:200px;background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#5e5a4a;">Tolerância de aspecto (tol)</label>
        <span id="sim06_ep08_vl_tol" style="font-family:monospace;font-weight:700;color:#26241d;">0.22</span>
      </div>
      <input id="sim06_ep08_sl_tol" style="width:100%;accent-color:#26241d;cursor:pointer;height:4px;" max="1.0" min="0.05" step="0.01" type="range" value="0.22">
    </div>
  </div>

  <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(240px, 1fr));gap:14px;margin-bottom:14px;">
    <!-- Canvas da Cena -->
    <div style="text-align:center;background:#fafaf7;border:1px solid #e9e3d3;padding:14px;border-radius:12px;">
      <div style="font-size:10px;font-weight:700;color:#8a8371;text-transform:uppercase;margin-bottom:10px;letter-spacing:0.04em;">Visualização da Cena (Matriz f)</div>
      <canvas id="sim06_ep08_canvas" width="260" height="260" style="border:1px solid #e4dcc8;border-radius:10px;background:#ffffff;margin:0 auto;display:block;"></canvas>
    </div>
    
    <!-- Lista de Candidatos -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;padding:14px;border-radius:12px;">
      <div style="font-size:10px;font-weight:700;color:#8a8371;text-transform:uppercase;margin-bottom:10px;text-align:center;letter-spacing:0.04em;">Componentes Conectados na Fila</div>
      <div id="sim06_ep08_lista" style="font-family:monospace;font-size:11px;display:flex;flex-direction:column;gap:8px;"></div>
    </div>
  </div>
  
  <!-- Console de Saída VPL -->
  <div id="sim06_ep08_debug" style="background:#fafaf7;border-radius:12px;padding:12px;border:1px solid #e9e3d3;font-family:monospace;font-size:11px;color:#26241d;text-align:center;"></div>
</div>

<script>
(function(){
  function initSim06Ep08(root){
    if(!root || root.dataset.sim06Ep08Init) return;
    root.dataset.sim06Ep08Init = "1";

    var formas = [
      {x: 145, y: 35,  w: 76, h: 76, tipo: "Componente QRCode Real", cor: "#cbd5e1", decodifica: true, padrao: "qr"},
      {x: 35,  y: 145, w: 55, h: 68, tipo: "Falso QRCode (Assimétrico)", cor: "#e2e8f0", decodifica: false, padrao: "falso_qr"},
      {x: 45,  y: 35,  w: 44, h: 44, tipo: "Círculo / Distrator", cor: "#f1f5f9", decodifica: false, padrao: "circulo"},
      {x: 160, y: 175, w: 68, h: 26, tipo: "Retângulo Distrator", cor: "#e2e8f0", decodifica: false, padrao: "retangulo"},
      {x: 65,  y: 220, w: 14, h: 14, tipo: "Ruído Isolado", cor: "#f8fafc", decodifica: false, padrao: "ruido"}
    ];

    var slA = root.querySelector('#sim06_ep08_sl_amin');
    var vlA = root.querySelector('#sim06_ep08_vl_amin');
    var slT = root.querySelector('#sim06_ep08_sl_tol');
    var vlT = root.querySelector('#sim06_ep08_vl_tol');
    var canvas = root.querySelector('#sim06_ep08_canvas');
    var ctx = canvas.getContext('2d');
    var lista = root.querySelector('#sim06_ep08_lista');
    var dbg = root.querySelector('#sim06_ep08_debug');

    function desenhaForma(f, estado){
      ctx.save();
      
      var corBorda = '#94a3b8';
      if (estado === 'candidato_ok') corBorda = '#10b981';
      if (estado === 'candidato_falhou') corBorda = '#f43f5e';
      if (estado === 'rejeitado') corBorda = '#cbd5e1';

      ctx.lineWidth = (estado === 'candidato_ok' || estado === 'candidato_falhou') ? 3 : 1.5;
      ctx.strokeStyle = corBorda;

      if (estado === 'rejeitado') {
        ctx.fillStyle = '#f8fafc';
      } else {
        if(f.padrao === 'qr') ctx.fillStyle = '#e2e8f0';
        else if(f.padrao === 'retangulo') ctx.fillStyle = '#fffbeb';
        else if(f.padrao === 'falso_qr') ctx.fillStyle = '#f0f9ff';
        else ctx.fillStyle = '#fdf4ff';
      }

      if(f.padrao === 'circulo'){
        ctx.beginPath();
        ctx.arc(f.x + f.w/2, f.y + f.h/2, f.w/2, 0, 2 * Math.PI);
        ctx.fill(); ctx.stroke();
      } else {
        ctx.fillRect(f.x, f.y, f.w, f.h);
        ctx.strokeRect(f.x, f.y, f.w, f.h);
        
        if(f.padrao === 'qr' || f.padrao === 'falso_qr'){
          var c = f.w / 5;
          ctx.fillStyle = '#ffffff';
          [[f.x + 3, f.y + 3], [f.x + f.w - c - 3, f.y + 3], [f.x + 3, f.y + f.h - c - 3]].forEach(function(p){
            ctx.fillRect(p[0], p[1], c, c);
            ctx.strokeRect(p[0], p[1], c, c);
          });
          
          ctx.fillStyle = (f.padrao === 'qr') ? '#334155' : '#64748b';
          [[f.x + 5, f.y + 5], [f.x + f.w - c + 1, f.y + 5], [f.x + 5, f.y + f.h - c + 1]].forEach(function(p){
            ctx.fillRect(p[0], p[1], c - 4, c - 4);
          });
        }
      }
      ctx.restore();
    }

    function render(){
      var amin = parseInt(slA.value, 10);
      var tol = parseFloat(slT.value);
      vlA.textContent = amin;
      vlT.textContent = tol.toFixed(2);

      ctx.clearRect(0, 0, canvas.width, canvas.height);
      ctx.fillStyle = '#ffffff';
      ctx.fillRect(0, 0, canvas.width, canvas.height);

      var candidatos = formas.map(function(f){
        var area = f.w * f.h;
        var aspecto = f.w / f.h;
        var passaArea = area > amin;
        var passaAspecto = Math.abs(aspecto - 1.0) <= tol;
        return {f: f, area: area, aspecto: aspecto, passa: passaArea && passaAspecto};
      }).sort(function(a, b){ return b.area - a.area; });

      lista.innerHTML = '';
      var encontrado = null;
      var flagParada = false;

      candidatos.forEach(function(c){
        var estado, texto, bgBox, txBox;
        
        if(!c.passa){
          estado = 'rejeitado';
          texto = 'REJEITADO (Área = ' + c.area + ' px&sup2;, Aspeto = ' + c.aspecto.toFixed(2) + ')';
          bgBox = '#f1f5f9';
          txBox = '#94a3b8';
        } else if(flagParada){
          estado = 'rejeitado';
          texto = 'FILA INTERROMPIDA (Critério de Parada Ativo)';
          bgBox = '#f8fafc';
          txBox = '#cbd5e1';
        } else if(c.f.decodifica){
          estado = 'candidato_ok';
          texto = 'SUCESSO: DECODIFICADO &#10004;';
          bgBox = '#ecfdf5';
          txBox = '#059669';
          encontrado = c.f;
          flagParada = true;
        } else {
          estado = 'candidato_falhou';
          texto = 'GEOMETRIA OK &rarr; FALHA NA DECODIFICAÇÃO &#10008;';
          bgBox = '#fff5f5';
          txBox = '#e11d48';
        }
        
        desenhaForma(c.f, estado);
        
        var div = document.createElement('div');
        div.style.cssText = 'padding:8px 10px;border-radius:8px;background:' + bgBox + ';border:1px solid #edf2f7;color:' + txBox + ';display:flex;flex-direction:column;gap:2px;';
        
        var nameSpan = document.createElement('strong');
        nameSpan.style.fontSize = '11px';
        nameSpan.textContent = c.f.tipo + ' (' + c.area + ' px²)';
        
        var statusSpan = document.createElement('span');
        statusSpan.style.fontSize = '10px';
        statusSpan.style.opacity = '0.9';
        statusSpan.innerHTML = texto;

        div.appendChild(nameSpan);
        div.appendChild(statusSpan);
        lista.appendChild(div);
      });

      if (encontrado) {
        dbg.style.backgroundColor = '#eafaf1';
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.color = '#04342C';
        dbg.innerHTML = '<div style="text-align:left;font-weight:700;color:#04342C;margin-bottom:4px;">&#128994; SAÍDA PADRÃO (VPL):</div>' +
                        'y=' + encontrado.y + ' x=' + encontrado.x + ' h=' + encontrado.h + ' w=' + encontrado.w + '<br>' +
                        '<span style="color:#04342C;font-weight:700;">"EP06_08 - PDI-VC | Parabens! Voce decodificou este QR Code!"</span>';
      } else {
        dbg.style.backgroundColor = '#fdecea';
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.color = '#c0392b';
        dbg.innerHTML = '<div style="text-align:left;font-weight:700;color:#c0392b;margin-bottom:4px;">&#128308; SAÍDA PADRÃO (VPL):</div>' +
                        'QRCODE_NAO_ENCONTRADO';
      }
    }

    slA.addEventListener('input', render);
    slT.addEventListener('input', render);
    render();
  }
  
  function tryInitSim06Ep08(){
    var root = document.getElementById('sim06_ep08');
    if(root) initSim06Ep08(root); else setTimeout(tryInitSim06Ep08, 200);
  }
  tryInitSim06Ep08();
})();
</script>
</div>
""")

In [24]:
%%writefile EP06_08.py
# Código Python

Overwriting EP06_08.py


In [25]:
TestSuite("EP06_08.py").run()